# ✅ Solutions — Session 1: Series & DataFrames

Worked answers to every exercise plus the **Student Gradebook** mini project.
Each solution is self-contained and uses pandas 3.x idioms only.

In [1]:
import io

import numpy as np
import pandas as pd

## Exercise 1 — Series from a list with a name

**(Easy)** Build the Series with `name=`. Passing `name=` labels it; the index
defaults to `RangeIndex(0, 4)`.

In [2]:
temps = pd.Series([12, 15, 9, 18], name="temps")
print("name :", temps.name)
print("index:", list(temps.index))
print("dtype:", temps.dtype)

name : temps
index: [0, 1, 2, 3]
dtype: int64


## Exercise 2 — Series from a dict, then a custom index

**(Easy)** The **dict keys** become the index. Rebuilding with `index=` replaces those labels
while keeping the data in the original order.

In [3]:
fruit = pd.Series({"apples": 3, "bananas": 5, "cherries": 7})
print("default index:", list(fruit.index))

fruit_renamed = pd.Series(fruit.to_numpy(), index=["A", "B", "C"], name="fruit")
fruit_renamed

default index: ['apples', 'bananas', 'cherries']


A    3
B    5
C    7
Name: fruit, dtype: int64

## Exercise 3 — DataFrame from a dict of lists

**(Medium, coding)** Each list is one column; every list must be the same length.

In [4]:
products = pd.DataFrame(
    {
        "product": ["Widget", "Gadget", "Gizmo"],
        "price": [9.99, 19.99, 4.50],
        "units": [120, 45, 300],
    }
)
print("shape  :", products.shape)
print("columns:", list(products.columns))
print(products.dtypes)

shape  : (3, 3)
columns: ['product', 'price', 'units']
product        str
price      float64
units        int64
dtype: object


## Exercise 4 — Same table three ways

**(Medium)** A list of dicts and `read_csv(io.StringIO(...))` should reproduce the dict-of-lists table.
We compare with `DataFrame.equals`, which checks values and dtypes.

In [5]:
list_of_dicts = [
    {"product": "Widget", "price": 9.99, "units": 120},
    {"product": "Gadget", "price": 19.99, "units": 45},
    {"product": "Gizmo", "price": 4.50, "units": 300},
]
from_dicts = pd.DataFrame(list_of_dicts)

csv_text = """product,price,units
Widget,9.99,120
Gadget,19.99,45
Gizmo,4.50,300
"""
from_csv = pd.read_csv(io.StringIO(csv_text))

print("from list of dicts == dict of lists :", from_dicts.equals(products))
print("from CSV           == dict of lists :", from_csv.equals(products))
from_csv

from list of dicts == dict of lists : True
from CSV           == dict of lists : True


,product,price,units
0,Widget,9.99,120
1,Gadget,19.99,45
2,Gizmo,4.50,300


## Exercise 5 — Chained assignment vs `.loc`

**(Advanced, conceptual)** Under Copy-on-Write, `df["score"][0] = 99` writes into a *temporary* object returned by
`df["score"]`; the parent frame is never updated (and pandas warns). The idiomatic,
CoW-safe form names the row and column explicitly: `df.loc[0, "score"] = 99`.

In [6]:
demo = pd.DataFrame({"score": [10, 20, 30]})
demo.loc[0, "score"] = 99          # correct: single explicit indexer
print(demo)

   score
0     99
1     20
2     30


## 🚀 Mini Project — Student Gradebook

In [7]:
students = pd.DataFrame(
    {
        "student": ["Ana", "Bo", "Cara", "Dan", "Eve"],
        "math": [91, 78, 84, 65, 88],
        "science": [89, 82, 79, 70, 95],
    }
)

# Step 2 — row-wise average of the two subjects.
students["average"] = students[["math", "science"]].mean(axis=1)

# Step 3 — letter grades via a helper + Series.map.
def letter_grade(avg):
    if avg >= 90:
        return "A"
    if avg >= 80:
        return "B"
    if avg >= 70:
        return "C"
    return "F"


students["letter"] = students["average"].map(letter_grade)

# Step 4 — inspect.
print(students.head())
print()
print(students.describe())

  student  math  science  average letter
0     Ana    91       89     90.0      A
1      Bo    78       82     80.0      B
2    Cara    84       79     81.5      B
3     Dan    65       70     67.5      F
4     Eve    88       95     91.5      A

            math    science    average
count   5.000000   5.000000   5.000000
mean   81.200000  83.000000  82.100000
std    10.281051   9.565563   9.600781
min    65.000000  70.000000  67.500000
25%    78.000000  79.000000  80.000000
50%    84.000000  82.000000  81.500000
75%    88.000000  89.000000  90.000000
max    91.000000  95.000000  91.500000


In [8]:
# Step 5 — only student and average.
students[["student", "average"]]

,student,average
0,Ana,90.0
1,Bo,80.0
2,Cara,81.5
3,Dan,67.5
4,Eve,91.5


In [9]:
# Step 6 — class average and pass count.
print(f"Class average : {students['average'].mean():.2f}")
print(f"Students >= 70: {int((students['average'] >= 70).sum())} of {len(students)}")
print(students["letter"].value_counts().sort_index())

Class average : 82.10
Students >= 70: 4 of 5
letter
A    2
B    2
F    1
Name: count, dtype: int64
